## VOC 리모컨 사용성 주제 분류 파이프라인

### 대상 데이터
- **소스**: `kic_data_ods.buzzmetrix.buzzmetrix`
- **카테고리**: 07. 스마트 사용성 > 07-06. 리모컨 사용성
- **감성**: 긍정 (sc_measurement = 1)
- **모수**: 32,462건 (중복 memo 제거)

---

### Notebook 1: `topic_tagging_v4_260420`
> 초기 분류 + Phase 1 (전반적 긍정 재분류)

| 단계 | 셀 | 내용 | 산출물 |
|---|---|---|---|
| Config | 1-2 | imports, 테이블명, 상수 설정 | - |
| Helpers | 3 | clean_text, call_llm, save_table 등 | - |
| 규칙 프로필 생성 | 4-5 | buzzmetrix에서 감성별 800건 샘플 → LLM이 읽고 분류 규칙 자동 설계 (clue_keywords, generic_terms) | `rule_profile` 테이블 |
| 토픽 풀 구축 | 6-7 | LLM으로 7\~17개 주제 분류 체계 설계 | `topic_pool` 테이블 |
| 샘플 분류 | 8-9 | 1,000건 샘플 LLM 분류 + 소수토픽 병합 | `detail` / `summary` |
| 전체 모수 분류 | 10-15 | 나머지 31,462건 배치 분류 (checkpoint/resume) | `detail_full` / `summary_full` |
| **Phase 1** | 16 | "전반적 긍정" 6,127건 → 초단문 67건만 유지, 나머지 LLM 재분류 | `detail_full_reoverall` / `summary_full_reoverall` |

**Phase 1 결과**: 전반적 긍정 18.9% → 0.2% 축소, **기타 14.3% (4,627건)** 발생

---

### Notebook 2: `topic_reclassify_etc` (현재 노트북)
> Phase 2 (기타 재분류) + Phase 3 (오분류 재분류) + Phase 4 (토픽 정리)

| 단계 | 셀 | 내용 | 입력 → 출력 |
|---|---|---|---|
| Config | 1 | 소스/저장 테이블, LLM 엔드포인트 (gpt-5-4-mini) | - |
| Helpers | 2 | clean_text, call_llm, save_table | - |
| 데이터 로드 | 3 | Phase 1 결과에서 기타 4,627건 추출 | `reoverall` → etc_df |
| **Phase 2a** | 4 | 규칙 기반: 리모컨+긍정+filler만 → "전반적 긍정" | 규칙으로 분리 |
| **Phase 2b** | 6 | LLM: 나머지 기타 → 전반적긍정 / 오분류 / 구체적토픽 | 배치 50건씩 |
| 중간 저장 | 8 | 비기타 + 규칙분류 + LLM재분류 병합 | `reoverall_v2` 테이블 |
| **Phase 3** | 10 | 오분류 1,006건 → 리모컨 평가 기준으로 재분류 | `reoverall_v2` 덮어쓰기 |
| **Phase 4** | 12 | 호환성만족 → 설정·연동·인식용이, 10건 미만 → 기타 | `reoverall_v2` 덮어쓰기 |

---

### 테이블 계보

```
[Notebook 1]
buzzmetrix (raw)
  │
  ├─ rule_profile          규칙 프로필 (clue_keywords, generic_terms)
  ├─ topic_pool             토픽 분류 체계 (14개 주제)
  ├─ detail / summary       샘플 1,000건 분류
  ├─ detail_full            전체 32,462건 분류
  └─ detail_full_reoverall  Phase 1: 전반적긍정 재분류

[Notebook 2]
detail_full_reoverall (input)
  │
  └─ detail_full_reoverall_v2   Phase 2+3+4: 기타/오분류 재분류 + 토픽 정리
```

### 최종 토픽 분포 (19개 주제)

| 토픽 | 건수 | 비중 |
|---|---|---|
| 직관적 조작·쉬운 사용 | 7,725 | 23.8% |
| 포인터·마우스·휠 조작 | 3,812 | 11.7% |
| 통합 제어·범용 리모컨 | 3,543 | 10.9% |
| 전반적 긍정 | 3,118 | 9.6% |
| 앱 바로가기·전용 버튼 | 2,622 | 8.1% |
| 충전식·태양광·배터리 불필요 | 2,120 | 6.5% |
| 그립감·크기·무게·디자인 | 1,672 | 5.2% |
| 버튼 구성·배치·단순함 | 1,548 | 4.8% |
| 반응속도·응답성 | 1,386 | 4.3% |
| 학습 용이·적응 후 편리 | 1,207 | 3.7% |
| 음성 제어 | 1,171 | 3.6% |
| 설정·연동·인식 용이 | 771 | 2.4% |
| 기능만족 | 577 | 1.8% |
| 백라이트·조명 편의 | 561 | 1.7% |
| 오분류 | 295 | 0.9% |
| 기술만족 | 110 | 0.3% |
| 휴대폰 리모컨 대체 | 110 | 0.3% |
| 수량만족 | 100 | 0.3% |
| 기타 | 14 | 0.04% |

In [0]:
# ─── 기타 재분류 노트북 ─────────────────────────────────────────────────
# topic_tagging_v4_260420 결과에서 "기타" 행만 추출하여 재분류

from __future__ import annotations

import json
import re
import time
from typing import Any, Dict, List, Optional

import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql import Window

# ─── Config ───────────────────────────────────────────────────────────────

SAVE_DB = "sandbox.z_jungryo_lee"
PROMPT_VERSION = "category_topic_dynamic_rules_v1"
ENDPOINT = "databricks-gpt-5-4-mini"

# 소스 테이블 (Phase 1 재분류 완료 결과)
SOURCE_DETAIL_TABLE = f"{SAVE_DB}.category_topic_detail_full_reoverall_{PROMPT_VERSION}"
TOPIC_POOL_TABLE = f"{SAVE_DB}.category_topic_pool_{PROMPT_VERSION}"
RULE_PROFILE_TABLE = f"{SAVE_DB}.category_topic_rule_profile_{PROMPT_VERSION}"

# 저장 테이블
OUTPUT_DETAIL_TABLE = f"{SAVE_DB}.category_topic_detail_full_reoverall_v2_{PROMPT_VERSION}"
OUTPUT_SUMMARY_TABLE = f"{SAVE_DB}.category_topic_summary_full_reoverall_v2_{PROMPT_VERSION}"

TARGET_SC_MEASUREMENT = 1
MAX_RETRIES = 3
BACKOFF_BASE = 1.8
MAX_TOKENS = 2200
RATE_LIMIT_SECONDS = 0.4
BATCH_SIZE = 50

print(f"[CONFIG] source={SOURCE_DETAIL_TABLE}")
print(f"[CONFIG] output={OUTPUT_DETAIL_TABLE}")
print(f"[CONFIG] endpoint={ENDPOINT}")

In [0]:
# ─── Helper Functions ─────────────────────────────────────────────────────

def clean_text(x: Any) -> str:
    return "" if x is None else re.sub(r"\s+", " ", str(x)).strip()


def extract_json(text: str) -> Dict[str, Any]:
    text = clean_text(text)
    for candidate in [text]:
        try:
            return json.loads(candidate)
        except Exception:
            pass
    match = re.search(r"\{.*\}", text, flags=re.S)
    if match:
        candidate = match.group(0)
        try:
            return json.loads(candidate)
        except Exception:
            candidate = re.sub(r",\s*}", "}", candidate)
            candidate = re.sub(r",\s*]", "]", candidate)
            return json.loads(candidate)
    raise ValueError(f"Invalid JSON: {text[:1000]}")


def chunk_list(items: List[Any], size: int) -> List[List[Any]]:
    return [items[i:i + size] for i in range(0, len(items), size)]


def call_llm(messages: List[Dict[str, str]], max_tokens: int = MAX_TOKENS) -> Dict[str, Any]:
    from mlflow.deployments import get_deploy_client
    client = get_deploy_client("databricks")
    payload = {"messages": messages, "temperature": 0.0, "max_tokens": max_tokens}

    last_error: Optional[Exception] = None
    for attempt in range(MAX_RETRIES):
        resp = None
        try:
            print(f"[LLM CALL] attempt={attempt + 1}/{MAX_RETRIES}")
            resp = client.predict(endpoint=ENDPOINT, inputs=payload)
            if isinstance(resp, dict):
                if "choices" in resp and resp["choices"]:
                    return extract_json(resp["choices"][0]["message"]["content"])
                if "predictions" in resp and resp["predictions"]:
                    pred0 = resp["predictions"][0]
                    if isinstance(pred0, dict) and "content" in pred0:
                        return extract_json(pred0["content"])
                    if isinstance(pred0, str):
                        return extract_json(pred0)
                if "content" in resp:
                    return extract_json(resp["content"])
            if isinstance(resp, str):
                return extract_json(resp)
            raise ValueError(f"Unexpected response schema: {resp}")
        except Exception as exc:
            last_error = exc
            print(f"[LLM ERROR] attempt={attempt + 1}/{MAX_RETRIES}, error={repr(exc)}")
            if resp is not None:
                print(str(resp)[:2000])
            time.sleep(BACKOFF_BASE ** attempt)
    raise RuntimeError(f"LLM call failed: {repr(last_error)}")


def save_table(df, table_name: str) -> None:
    if spark.catalog.tableExists(table_name):
        print(f"[SAVE] overwrite -> {table_name}")
    else:
        print(f"[SAVE] create -> {table_name}")
    df.write.mode("overwrite").format("delta").saveAsTable(table_name)

In [0]:
# ─── 테이블 로드 & 기타 행 추출 ───────────────────────────────────────────

# 1) 기존 분류 결과 로드
detail_df = (
    spark.table(SOURCE_DETAIL_TABLE)
    .select("_row_id", "memo", "topic", "description", "classification_route")
    .withColumn("similarity_score", F.lit(None).cast("double"))
)

# 2) 토픽 풀 로드
topic_pool_df = spark.table(TOPIC_POOL_TABLE)

# 3) 규칙 프로필 → OVERALL_TOPIC_LABEL
rule_profile_row = (
    spark.table(RULE_PROFILE_TABLE)
    .where(F.col("sc_measurement") == F.lit(TARGET_SC_MEASUREMENT))
    .limit(1)
    .collect()[0]
)
OVERALL_TOPIC_LABEL = rule_profile_row["overall_topic_label"]

# 4) 기타 행 추출
etc_condition = (F.col("topic") == F.lit("기타")) | F.col("topic").like("기타(%")
etc_df = detail_df.where(etc_condition)
non_etc_df = detail_df.where(~etc_condition)

total_cnt = detail_df.count()
etc_cnt = etc_df.count()
non_etc_cnt = non_etc_df.count()

print(f"[전체] {total_cnt} rows")
print(f"[기타] {etc_cnt} rows ({etc_cnt/total_cnt*100:.1f}%)")
print(f"[비기타] {non_etc_cnt} rows")
print(f"[OVERALL_TOPIC_LABEL] {OVERALL_TOPIC_LABEL}")

display(etc_df.limit(20))

In [0]:
# ─── 규칙 기반: 리모컨 지칭 + 긍정 감성 + filler만 → "전반적 긍정" ──────

REMOTE_TERMS = [
    "remote", "remote control", "remotecontrol", "controller",
    "리모컨", "리모콘", "리모트", "컨트롤러", "조작기", "조종기",
    "magic remote", "one remote", "smart remote",
    "매직리모컨", "매직 리모컨", "원리모컨", "스마트리모컨",
]

REMOTE_POSITIVE_TERMS = [
    "good", "great", "nice", "best", "excellent", "love", "awesome",
    "amazing", "fantastic", "wonderful", "perfect", "superb", "cool",
    "좋아요", "좋다", "좋음", "좋아", "좋습니다", "좋네요",
    "최고", "만족", "괜찮아요", "괜찮다", "굿", "나이스", "훌륭", "대박",
]

FILLER_WORDS = {
    "is", "are", "was", "were", "the", "a", "an", "very", "so", "really",
    "quite", "pretty", "just", "also", "too", "it", "its", "this", "that",
    "my", "our", "i", "we", "be", "been", "has", "have", "had", "do",
    "정말", "진짜", "매우", "아주", "너무", "참", "되게", "엄청", "완전",
}


def is_remote_generic_positive(text: str) -> bool:
    """리모컨 지칭 단어 + 긍정 감성 단어 + filler만 있으면 True (구체적 이유 없음)"""
    memo = clean_text(text).lower()
    if not memo:
        return False

    has_remote = any(t in memo for t in REMOTE_TERMS)
    if not has_remote:
        return False

    has_positive = any(t in memo for t in REMOTE_POSITIVE_TERMS)
    if not has_positive:
        return False

    # 리모컨/긍정/filler 단어 제거 후 실질적 의미 단어가 남으면 False
    remaining = memo
    for term in sorted(REMOTE_TERMS, key=len, reverse=True):
        remaining = remaining.replace(term, " ")
    for term in sorted(REMOTE_POSITIVE_TERMS, key=len, reverse=True):
        remaining = remaining.replace(term, " ")

    words = re.findall(r"[a-zA-Z가-힣]+", remaining)
    meaningful = [w for w in words if w not in FILLER_WORDS and len(w) > 1]

    return len(meaningful) == 0


# 기타 행 분리
etc_rows = [r.asDict(recursive=True) for r in etc_df.toLocalIterator()]

generic_positive_rows = []
to_reclassify_rows = []

for row in etc_rows:
    if is_remote_generic_positive(row["memo"]):
        generic_positive_rows.append({
            "_row_id": row["_row_id"],
            "memo": row["memo"],
            "topic": OVERALL_TOPIC_LABEL,
            "description": "리모컨에 대한 구체적 이유 없는 일반 긍정",
            "classification_route": "etc_rule_generic_positive",
            "similarity_score": row.get("similarity_score"),
        })
    else:
        to_reclassify_rows.append(row)

print(f"[기타 전체] {len(etc_rows)}")
print(f"[규칙 → 전반적 긍정] {len(generic_positive_rows)}")
print(f"[LLM 재분류 대상] {len(to_reclassify_rows)}")



In [0]:
# 규칙 분류 예시 확인
if generic_positive_rows:
    print("\n[규칙 → 전반적 긍정 예시]")
    for r in generic_positive_rows[6:30]:
        print(f"  - {r['memo'][:80]}")

In [0]:
# ─── LLM 기타 재분류 (3가지 규칙: 전반적긍정 / 오분류 / 구체적토픽) ───────

phase2_topic_payload = [
    {"topic": r["topic"], "description": r["description"]}
    for r in topic_pool_df.orderBy("topic_order").toLocalIterator()
] + [
    {"topic": "오분류", "description": "리모컨이 아닌 다른 카테고리(화질, 음질, 화면, 배송 등)에 대한 칭찬이거나, 부정적/불만 문장"},
]


def build_reclassify_etc_batch_messages(batch_rows: List[Dict[str, Any]]) -> List[Dict[str, str]]:
    system = f"""You are a VOC classifier for TV remote-usability reviews.

These memos were classified as '기타' (Others). Reclassify each into exactly one category.

Rules:
1. '전반적 긍정': The memo ONLY says the remote is good with NO specific reason/feature/benefit.
   Examples: "remote is great", "리모컨 좋아요", "love the remote"

2. '오분류': The memo praises something OTHER than the remote (picture, sound, screen, delivery, etc.)
   OR the memo is negative/complaining about the remote.
   Examples: "화질이 정말 좋아요", "remote doesn't work", "great sound quality"

3. Specific topic: The memo hints at WHY the remote is good. Classify into:
   - An existing topic from the list if it fits
   - OR a new concise Korean label if no existing topic fits
     (e.g., "기능만족", "이용성만족", "수량만족", "기술만족", "호환성만족", "가성비만족")
   Even minimal clues count.
   Examples:
     "it has many built in smart features" → 앱 바로가기·전용 버튼 or 기능만족
     "remote control is very useful" → 직관적 조작·쉬운 사용
     "2 remote controls are useful" → 수량만족
     "The remote feels premium in hand" → 그립감·크기·무게·디자인
     "high tech remote" → 기술만족

Important:
- Prefer existing topics when a reasonable match exists.
- New labels must be concise Korean (2-5 chars + optional qualifier).
- explanation must be a short Korean sentence.

Return only JSON:
{{
  "results": [
    {{
      "row_id": "",
      "topic": "",
      "explanation": ""
    }}
  ]
}}
"""
    user = (
        "Allowed topics:\n"
        + json.dumps(phase2_topic_payload, ensure_ascii=False)
        + "\n\nMemos:\n"
        + json.dumps(
            [{"row_id": str(r["_row_id"]), "memo": clean_text(r["memo"])} for r in batch_rows],
            ensure_ascii=False,
        )
    )
    return [
        {"role": "system", "content": clean_text(system)},
        {"role": "user", "content": clean_text(user)},
    ]


# ─── 배치 분류 루프 ─────────────────────────────────────────────────────

reclassified_rows = []
total_batches = (len(to_reclassify_rows) + BATCH_SIZE - 1) // BATCH_SIZE

for batch_no, batch in enumerate(chunk_list(to_reclassify_rows, BATCH_SIZE), start=1):
    print(f"[BATCH] {batch_no}/{total_batches}, rows={len(batch)}")
    batch_result = call_llm(build_reclassify_etc_batch_messages(batch))
    result_map = {
        str(item.get("row_id")): item
        for item in batch_result.get("results", [])
        if isinstance(item, dict)
    }

    for row in batch:
        mapped = result_map.get(str(row["_row_id"]), {})
        new_topic = clean_text(mapped.get("topic"))
        new_expl = clean_text(mapped.get("explanation"))

        if not new_topic:
            new_topic = "기타"

        reclassified_rows.append({
            "_row_id": row["_row_id"],
            "memo": row["memo"],
            "topic": new_topic,
            "description": new_expl or "기타 재분류",
            "classification_route": "etc_recheck",
            "similarity_score": row.get("similarity_score"),
        })

    time.sleep(RATE_LIMIT_SECONDS)

print(f"\n[LLM 재분류 완료] {len(reclassified_rows)} rows")

In [0]:
# ─── 최종 병합 & 저장 ────────────────────────────────────────────────────

# 규칙 분류 → 전반적 긍정
generic_positive_df = (
    spark.createDataFrame(pd.DataFrame(generic_positive_rows))
    if generic_positive_rows
    else spark.createDataFrame(
        [], "_row_id long, memo string, topic string, description string, classification_route string, similarity_score double"
    )
)

# LLM 재분류 결과
reclassified_df = (
    spark.createDataFrame(pd.DataFrame(reclassified_rows))
    if reclassified_rows
    else spark.createDataFrame(
        [], "_row_id long, memo string, topic string, description string, classification_route string, similarity_score double"
    )
)

# 최종 병합: 비기타 + 규칙분류(전반적긍정) + LLM재분류
final_detail_df = non_etc_df.unionByName(generic_positive_df).unionByName(reclassified_df)

# 요약 통계
final_summary_df = (
    final_detail_df.groupBy("topic")
    .agg(F.count("*").alias("review_count"))
    .withColumn("total_count", F.sum("review_count").over(Window.partitionBy()))
    .withColumn("review_share", F.round(F.col("review_count") / F.col("total_count"), 4))
    .withColumn("review_share_pct", F.round(F.col("review_share") * 100, 2))
    .orderBy(F.desc("review_count"), "topic")
)

# 저장
save_table(final_detail_df, OUTPUT_DETAIL_TABLE)
save_table(final_summary_df, OUTPUT_SUMMARY_TABLE)

print(f"\n[최종 결과] 전체 {final_detail_df.count()} rows")
print("\n[SUMMARY]")
display(final_summary_df)

print("\n[DETAIL 샘플]")
display(final_detail_df.limit(30))

In [0]:
display(reclassified_df)

In [0]:
# ─── Phase 3: 오분류 재분류 ───────────────────────────────────────────────
# 다른 카테고리와 함께 리모컨을 언급한 리뷰 → 리모컨 평가 기준으로 재분류

# 1) 오분류 행 추출 (OUTPUT_DETAIL_TABLE에서 새로 로드)
v2_detail_df = (
    spark.table(OUTPUT_DETAIL_TABLE)
    .select("_row_id", "memo", "topic", "description", "classification_route")
    .withColumn("similarity_score", F.lit(None).cast("double"))
)

misclass_df = v2_detail_df.where(F.col("topic") == F.lit("오분류"))
non_misclass_df = v2_detail_df.where(F.col("topic") != F.lit("오분류"))

misclass_rows = [r.asDict(recursive=True) for r in misclass_df.toLocalIterator()]
print(f"[오분류 전체] {len(misclass_rows)} rows")

# 2) 오분류 재분류 프롬프트
phase3_topic_payload = [
    {"topic": r["topic"], "description": r["description"]}
    for r in topic_pool_df.orderBy("topic_order").toLocalIterator()
] + [
    {"topic": "전반적 긍정", "description": "리모컨이 좋다고만 언급하고 구체적 이유/기능은 없음"},
    {"topic": "오분류", "description": "리모컨과 전혀 관련 없는 내용이거나, 리모컨에 대해 부정적인 문장"},
]


def build_reclassify_misclass_messages(batch_rows: List[Dict[str, Any]]) -> List[Dict[str, str]]:
    system = f"""You are a VOC classifier for TV remote-usability reviews.

These memos were classified as '오분류' because they mention multiple product categories (picture, sound, etc.) alongside the remote.
Your job: focus ONLY on how the writer evaluated the REMOTE CONTROL part, and classify accordingly.

Rules:
1. If the memo mentions the remote alongside other features (picture, sound, etc.),
   extract ONLY the remote-related sentiment and classify based on that.

2. '전반적 긍정': The remote is mentioned positively but with NO specific reason about the remote.
   Examples:
   - "sound and a remote. The tv is very good!" → remote is just listed as good → 전반적 긍정
   - "Great picture, features, flexibility, remote and performance." → remote listed among good things → 전반적 긍정
   - "I enjoy the sound, colors, remote and all the great features" → remote just listed → 전반적 긍정

3. Specific topic: The memo hints at WHY the remote is specifically good. Use existing topics or new labels.
   Examples:
   - "Love the picture quality and the features on the remote." → remote features → 기능만족
   - "Absolutely love the picture quality, remote functionality" → remote functionality → 기능만족
   - "Very happy with the clarity of the picture and remote features" → remote features → 기능만족
   - "Great picture and easy to use remote" → easy to use → 직관적 조작·쉬운 사용
   - "Nice screen and responsive remote" → responsive → 반응속도·응답성

4. '오분류': ONLY if the memo has absolutely NO mention of the remote,
   OR the remote is mentioned negatively/as a complaint.

Important:
- Prefer existing topics when possible.
- New labels must be concise Korean.
- explanation must be a short Korean sentence about the REMOTE part only.

Return only JSON:
{{
  "results": [
    {{
      "row_id": "",
      "topic": "",
      "explanation": ""
    }}
  ]
}}
"""
    user = (
        "Allowed topics:\n"
        + json.dumps(phase3_topic_payload, ensure_ascii=False)
        + "\n\nMemos:\n"
        + json.dumps(
            [{"row_id": str(r["_row_id"]), "memo": clean_text(r["memo"])} for r in batch_rows],
            ensure_ascii=False,
        )
    )
    return [
        {"role": "system", "content": clean_text(system)},
        {"role": "user", "content": clean_text(user)},
    ]


# 3) 배치 재분류
misclass_reclassified = []
total_batches = (len(misclass_rows) + BATCH_SIZE - 1) // BATCH_SIZE

for batch_no, batch in enumerate(chunk_list(misclass_rows, BATCH_SIZE), start=1):
    print(f"[BATCH] {batch_no}/{total_batches}, rows={len(batch)}")
    batch_result = call_llm(build_reclassify_misclass_messages(batch))
    result_map = {
        str(item.get("row_id")): item
        for item in batch_result.get("results", [])
        if isinstance(item, dict)
    }

    for row in batch:
        mapped = result_map.get(str(row["_row_id"]), {})
        new_topic = clean_text(mapped.get("topic"))
        new_expl = clean_text(mapped.get("explanation"))

        if not new_topic:
            new_topic = "오분류"

        misclass_reclassified.append({
            "_row_id": row["_row_id"],
            "memo": row["memo"],
            "topic": new_topic,
            "description": new_expl or "오분류 재분류",
            "classification_route": "misclass_recheck",
            "similarity_score": row.get("similarity_score"),
        })

    time.sleep(RATE_LIMIT_SECONDS)

print(f"\n[오분류 재분류 완료] {len(misclass_reclassified)} rows")

# 4) 결과 병합 & OUTPUT 테이블 덮어쓰기
misclass_reclass_df = (
    spark.createDataFrame(pd.DataFrame(misclass_reclassified))
    .withColumn("similarity_score", F.lit(None).cast("double"))
    if misclass_reclassified
    else spark.createDataFrame(
        [], "_row_id long, memo string, topic string, description string, classification_route string, similarity_score double"
    )
)

final_detail_v3_df = non_misclass_df.unionByName(misclass_reclass_df)

final_summary_v3_df = (
    final_detail_v3_df.groupBy("topic")
    .agg(F.count("*").alias("review_count"))
    .withColumn("total_count", F.sum("review_count").over(Window.partitionBy()))
    .withColumn("review_share", F.round(F.col("review_count") / F.col("total_count"), 4))
    .withColumn("review_share_pct", F.round(F.col("review_share") * 100, 2))
    .orderBy(F.desc("review_count"), "topic")
)

save_table(final_detail_v3_df, OUTPUT_DETAIL_TABLE)
save_table(final_summary_v3_df, OUTPUT_SUMMARY_TABLE)

print(f"\n[최종 결과] 전체 {final_detail_v3_df.count()} rows")
print("\n[UPDATED SUMMARY]")
display(final_summary_v3_df)

In [0]:
display(misclass_reclass_df)

In [0]:
# ─── Phase 4: 토픽 정리 ───────────────────────────────────────────────────
MIN_TOPIC_COUNT = 10

# 1) 토픽 이름 변경: 호환성만족 → 설정·연동·인식 용이
v3_detail_df = (
    spark.table(OUTPUT_DETAIL_TABLE)
    .select("_row_id", "memo", "topic", "description", "classification_route")
    .withColumn("similarity_score", F.lit(None).cast("double"))
)

renamed_df = v3_detail_df.withColumn(
    "topic",
    F.when(F.col("topic") == F.lit("호환성만족"), F.lit("설정·연동·인식 용이"))
    .otherwise(F.col("topic"))
)

# 2) 10건 미만 토픽 → 기타
stats = renamed_df.groupBy("topic").agg(F.count("*").alias("cnt")).collect()
small_topics = [
    r["topic"] for r in stats
    if r["cnt"] < MIN_TOPIC_COUNT and r["topic"] not in ("오분류",)
]

print(f"[10건 미만 토픽 → 기타] {small_topics}")

cleaned_df = renamed_df.withColumn(
    "topic",
    F.when(F.col("topic").isin(small_topics), F.lit("기타"))
    .otherwise(F.col("topic"))
)

# 3) 요약 & 저장
cleaned_summary_df = (
    cleaned_df.groupBy("topic")
    .agg(F.count("*").alias("review_count"))
    .withColumn("total_count", F.sum("review_count").over(Window.partitionBy()))
    .withColumn("review_share", F.round(F.col("review_count") / F.col("total_count"), 4))
    .withColumn("review_share_pct", F.round(F.col("review_share") * 100, 2))
    .orderBy(F.desc("review_count"), "topic")
)

save_table(cleaned_df, OUTPUT_DETAIL_TABLE)
save_table(cleaned_summary_df, OUTPUT_SUMMARY_TABLE)

print(f"\n[최종 결과] {cleaned_df.count()} rows")
print("\n[CLEANED SUMMARY]")
display(cleaned_summary_df)

In [0]:
# 대분류 매핑 dict
TOPIC_TO_MAIN_CATEGORY = {
    "직관적 조작·쉬운 사용": "조작 편의성 · 사용 흐름",
    "버튼 구성·배치·단순함": "조작 편의성 · 사용 흐름",
    "반응속도·응답성": "조작 편의성 · 사용 흐름",
    "학습 용이·적응 후 편리": "조작 편의성 · 사용 흐름",
    "포인터·마우스·휠 조작": "조작 방식 다양성",
    "음성 제어": "조작 방식 다양성",
    "휴대폰 리모컨 대체": "조작 방식 다양성",
    "통합 제어·범용 리모컨": "통합 제어 · 연동성",
    "설정·연동·인식 용이": "통합 제어 · 연동성",
    "앱 바로가기·전용 버튼": "버튼 · 접근성 강화",
    "백라이트·조명 편의": "버튼 · 접근성 강화",
    "충전식·태양광·배터리 불필요": "전원 · 유지관리 편의",
    "그립감·크기·무게·디자인": "물리적 디자인 만족",
    "전반적 긍정": "전반적 만족",
    "기능만족": "기타 · 비핵심 / 오분류",
    "기술만족": "기타 · 비핵심 / 오분류",
    "수량만족": "기타 · 비핵심 / 오분류",
    "오분류": "기타 · 비핵심 / 오분류",
    "기타": "기타 · 비핵심 / 오분류",
}

# 대분류 컬럼 추가
cleaned_df = cleaned_df.withColumn(
    "main_category",
    F.when(
        F.col("topic").isin(list(TOPIC_TO_MAIN_CATEGORY.keys())),
        F.create_map(*[item for k, v in TOPIC_TO_MAIN_CATEGORY.items() for item in (F.lit(k), F.lit(v))])[F.col("topic")]
    ).otherwise(F.lit("기타 · 비핵심 / 오분류"))
)

display(cleaned_df.limit(30))

In [0]:
# ─── main_category 포함 최종 저장 ──────────────────────────────────────

cleaned_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(OUTPUT_DETAIL_TABLE)
print(f"[SAVE] overwrite (schema) -> {OUTPUT_DETAIL_TABLE}")

# 대분류별 소계 계산
main_cat_totals = (
    cleaned_df.groupBy("main_category")
    .agg(F.count("*").alias("main_cat_total"))
)

cleaned_summary_with_main_df = (
    cleaned_df.groupBy("main_category", "topic")
    .agg(F.count("*").alias("review_count"))
    .withColumn("total_count", F.sum("review_count").over(Window.partitionBy()))
    .withColumn("review_share", F.round(F.col("review_count") / F.col("total_count"), 4))
    .withColumn("review_share_pct", F.round(F.col("review_share") * 100, 2))
    .join(main_cat_totals, on="main_category")
    .withColumn(
        "_is_etc",
        F.when(F.col("main_category") == F.lit("기타 · 비핵심 / 오분류"), F.lit(1)).otherwise(F.lit(0))
    )
    .orderBy(
        F.asc("_is_etc"),
        F.desc("main_cat_total"),
        F.desc("review_count"),
    )
    .drop("_is_etc", "main_cat_total")
)

cleaned_summary_with_main_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(OUTPUT_SUMMARY_TABLE)
print(f"[SAVE] overwrite (schema) -> {OUTPUT_SUMMARY_TABLE}")

print(f"[최종 저장 완료] detail={OUTPUT_DETAIL_TABLE}, summary={OUTPUT_SUMMARY_TABLE}")
print(f"[전체] {cleaned_df.count()} rows\n")
display(cleaned_summary_with_main_df)